# TreeProduct

[Codility task](https://app.codility.com/programmers/trainings/2/tree_product/) - Exercise 2, 2016 Contest

`N + 1` guard posts (numbered `0..N`) are connected by `N` bridges forming a tree (`A[K]` - `B[K]`). The Elves destroy **at most two** bridges and want to maximise `X * Y * Z` (three areas), `X * Y` (two areas) or `N + 1` (no cut), whichever is the largest. `solution(A, B)` returns it as a **string**.

| A | B | result |
|---|---|--------|
| `[0, 1, 1, 3, 3, 6, 7]` | `[1, 2, 3, 4, 5, 3, 5]` | `"18"` (areas of 3, 3 and 2 posts) |
| `[0, 1]` | `[1, 2]` | `"3"` (no cut) |

Assumptions: `N` in `[1..50,000]`, and every post is at most `900` bridges away from post 0.

## Idea

Root the tree at post 0 and let `s[v]` be the number of posts in the subtree of `v`; cutting the bridge above `v` separates `s[v]` posts. With `T = N + 1`:

- **no cut**: `T`; **one cut** above `a`: `s[a] * (T - s[a])`;
- **two cuts**, above `a` and above `b`, in one of two shapes:
  - **nested** (`a` is below `b`): areas `s[a]`, `s[b] - s[a]` and `T - s[b]`;
  - **unrelated** (neither is an ancestor of the other): areas `s[a]`, `s[b]` and `T - s[a] - s[b]`.

*Nested pairs.* For every node walk up its ancestors; the depth is at most 900, so the total work is O(N * depth). It is vectorised with NumPy: after `k` steps all nodes have their `k`-th ancestor computed at once.

*Unrelated pairs.* For a fixed `a`, `f(s[b]) = s[b] * (T - s[a] - s[b])` is a concave parabola in `s[b]`, so the best partner is the one whose size is closest to `(T - s[a]) / 2` from below or from above. Which nodes are unrelated to `a`? During a DFS, the nodes already **finished** when `a` is entered are exactly the unrelated nodes visited earlier (an ancestor is not finished yet, a descendant not yet started). Keep their sizes in a sorted list; at the entry of `a` look up the predecessor and the successor of `(T - s[a]) / 2` with `bisect`. Each unrelated pair is seen once, from its later node, and the optimum is found because the best partner of the later node is always among the earlier ones.

Time O(N log N + N * depth), memory O(N). The result is returned as `str`.

In [ ]:
from bisect import bisect_left, bisect_right, insort

import numpy as np


def solution(A, B):
    T = len(A) + 1                                         # number of posts
    adjacency = [[] for _ in range(T)]
    for a, b in zip(A, B):
        adjacency[a].append(b)
        adjacency[b].append(a)

    parent = [-1] * T                                       # BFS from post 0
    order = [0]
    for v in order:
        for w in adjacency[v]:
            if w != parent[v]:
                parent[w] = v
                order.append(w)
    children = [[] for _ in range(T)]
    for v in order[1:]:
        children[parent[v]].append(v)

    size = [1] * T
    for v in reversed(order[1:]):
        size[parent[v]] += size[v]

    best = T                                                # no bridge destroyed
    for v in range(1, T):                                   # one bridge destroyed
        best = max(best, size[v] * (T - size[v]))

    # nested pairs: b is a proper ancestor of a (and not the root, which has no bridge above it)
    par = np.array([0 if p < 0 else p for p in parent], dtype=np.int64)
    sz = np.array(size, dtype=np.int64)
    a = np.arange(1, T)
    anc = par[a]
    keep = anc != 0
    a, anc = a[keep], anc[keep]
    sa = sz[a]
    while a.size:
        sb = sz[anc]
        best = max(best, int((sa * (sb - sa) * (T - sb)).max()))
        anc = par[anc]
        keep = anc != 0
        a, anc, sa = a[keep], anc[keep], sa[keep]

    # unrelated pairs: DFS; the finished nodes are the unrelated ones visited before
    finished = []                                           # sorted sizes of finished non-root nodes
    stack = [0]
    while stack:
        v = stack.pop()
        if v >= 0:                                          # entering v
            if v != 0:
                s = size[v]
                rest = T - s
                low = bisect_right(finished, rest // 2)     # largest finished size <= rest / 2
                if low:
                    b = finished[low - 1]
                    best = max(best, s * b * (rest - b))
                high = bisect_left(finished, (rest + 1) // 2)    # smallest finished size >= rest / 2
                if high < len(finished):
                    b = finished[high]
                    best = max(best, s * b * (rest - b))
            stack.append(~v)
            stack.extend(children[v])
        elif ~v != 0:                                       # leaving a non-root node
            insort(finished, size[~v])
    return str(best)

## Official examples

In [ ]:
assert solution([0, 1, 1, 3, 3, 6, 7], [1, 2, 3, 4, 5, 3, 5]) == "18"
assert solution([0, 1], [1, 2]) == "3"
print("examples OK")

## Edge cases

In [ ]:
assert solution([0], [1]) == "2"                                   # two posts: 1 * 1 = 1 < 2, so no cut
assert solution([0, 1], [1, 2]) == "3"                             # path of 3: cutting gives at most 2
assert solution([0, 1, 2], [1, 2, 3]) == "4"                       # path of 4: 2 * 2 = 4, 1 * 1 * 2 = 2, no cut = 4
assert solution([0, 1, 2, 3], [1, 2, 3, 4]) == "6"                 # path of 5: 2 * 2 * 1 = 4, 2 * 3 = 6
assert solution([0, 1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]) == "12"    # path of 7: 2 * 2 * 3 = 12
assert solution([0, 0, 0], [1, 2, 3]) == "4"                       # star with 3 leaves: 1*3 = 3, 1*1*2 = 2, no cut = 4
assert solution([1, 2], [0, 0]) == "3"                             # edges given "backwards"
assert solution([0] * 50_000, list(range(1, 50_001))) == "50001"  # star: cutting leaves only loses (1 * 50000, 1 * 1 * 49999)
print("edge cases OK")

## References

1. **Every pair of bridges** (`N <= 9`): the literal definition. Remove each choice of at most two bridges, compute the sizes of the resulting components with a union-find, and take the maximum product.
2. **Every pair of nodes** (`N` up to 150): the nested/unrelated formulas evaluated for *all* pairs of non-root nodes with the ancestor test from an Euler tour, with no concavity trick and no NumPy. It covers deep trees (paths) and bigger shapes than the exhaustive version.

In [ ]:
import itertools
import random


def brute(A, B):
    T = len(A) + 1
    edges = list(zip(A, B))
    best = T
    for removed in range(1, 3):
        for cut in itertools.combinations(range(len(edges)), removed):
            leader = list(range(T))

            def find(x):
                while leader[x] != x:
                    leader[x] = leader[leader[x]]
                    x = leader[x]
                return x

            for i, (a, b) in enumerate(edges):
                if i not in cut:
                    leader[find(a)] = find(b)
            sizes = {}
            for v in range(T):
                sizes[find(v)] = sizes.get(find(v), 0) + 1
            product = 1
            for s in sizes.values():
                product *= s
            best = max(best, product)
    return str(best)


def pairs_reference(A, B):
    T = len(A) + 1
    adjacency = [[] for _ in range(T)]
    for a, b in zip(A, B):
        adjacency[a].append(b)
        adjacency[b].append(a)
    tin, tout, size, parent = [0] * T, [0] * T, [1] * T, [-1] * T
    clock = 0
    stack = [(0, iter(adjacency[0]))]
    tin[0] = 0
    while stack:
        v, it = stack[-1]
        for w in it:
            if w != parent[v]:
                parent[w] = v
                clock += 1
                tin[w] = clock
                stack.append((w, iter(adjacency[w])))
                break
        else:
            tout[v] = clock
            stack.pop()
            if parent[v] >= 0:
                size[parent[v]] += size[v]
    best = T
    nodes = range(1, T)
    for a in nodes:
        best = max(best, size[a] * (T - size[a]))
    for a, b in itertools.combinations(nodes, 2):
        if tin[a] <= tin[b] <= tout[a]:                     # a is an ancestor of b
            parts = (size[b], size[a] - size[b], T - size[a])
        elif tin[b] <= tin[a] <= tout[b]:                   # b is an ancestor of a
            parts = (size[a], size[b] - size[a], T - size[b])
        else:                                               # unrelated
            parts = (size[a], size[b], T - size[a] - size[b])
        best = max(best, parts[0] * parts[1] * parts[2])
    return str(best)


def random_tree(n_edges, rng, shape="random"):
    """Random tree on n_edges + 1 posts with random labels and random orientation of the bridges."""
    T = n_edges + 1
    edges = []
    for v in range(1, T):
        if shape == "random":
            p = rng.randrange(v)
        elif shape == "path":
            p = v - 1
        elif shape == "star":
            p = 0
        else:                                               # "deep": parent among the last few nodes
            p = rng.randrange(max(0, v - 3), v)
        edges.append((p, v))
    labels = list(range(T))
    rng.shuffle(labels)
    edges = [(labels[a], labels[b]) if rng.random() < 0.5 else (labels[b], labels[a]) for a, b in edges]
    rng.shuffle(edges)
    return [a for a, _ in edges], [b for _, b in edges]


rng = random.Random(12)
for _ in range(1500):
    n = rng.randint(1, 9)
    A, B = random_tree(n, rng, rng.choice(["random", "path", "star", "deep"]))
    assert solution(A, B) == brute(A, B), (A, B)
print("matches the bridge-pair brute force on 1500 random trees (N up to 9)")

for _ in range(500):
    n = rng.randint(1, 150)
    A, B = random_tree(n, rng, rng.choice(["random", "path", "star", "deep"]))
    assert solution(A, B) == pairs_reference(A, B), (A, B)
print("matches the all-pairs reference on 500 random trees (N up to 150)")

## Performance

`N = 50,000`. The statement bounds the depth by 900 (the nested part costs O(N * depth)): the worst case for it is a long path with all the other nodes hanging at its end.

In [ ]:
import time


def timed(name, A, B):
    start = time.perf_counter()
    result = solution(A, B)
    print(f"{name:34s} -> {result:>16} in {time.perf_counter() - start:.2f}s")


N = 50_000
rnd = random.Random(14)

A, B = random_tree(N, rnd, "random")
timed("random recursive tree", A, B)

A, B = [0] * N, list(range(1, N + 1))
timed("star", A, B)

D = 899                                                   # path of 899 bridges, every other node hangs at its end (depth 900)
A = list(range(D)) + [D] * (N - D)
B = list(range(1, D + 1)) + list(range(D + 1, N + 1))
timed("path of 900 + leaves at the end", A, B)

A = list(range(450)) + [450] * (N - 450)                   # broom
B = list(range(1, 451)) + list(range(451, N + 1))
timed("broom (path 450, then a star)", A, B)

spine = 899                                               # caterpillar: leaves on random spine nodes
A = list(range(spine)) + [rnd.randrange(spine) for _ in range(N - spine)]
B = list(range(1, spine + 1)) + list(range(spine + 1, N + 1))
timed("caterpillar (spine of 900)", A, B)

## Notes

- The shape of the cut decides the formula: *nested* cuts share the middle component `s[b] - s[a]`, *unrelated* cuts do not.
- Trick to avoid O(N^2): the best partner for a node is the size closest to half of what remains (concave parabola), found with a sorted list. "Finished before entering" in a DFS is an elegant way to select only the unrelated nodes.
- The depth bound of 900 is what makes the simple O(N * depth) walk over ancestors acceptable. On a path of 50,000 nodes it would take minutes.
- Results can be large products: Python integers do not overflow, and the statement asks for a string anyway.